## LanceDB  - a vector database for LLM applications

In [1]:
import lancedb

db = lancedb.connect(uri= "vector_database")
db

LanceDBConnection(uri='/Users/susannarokka/Desktop/NBI_year_2/Repo/AI_Engineering_OPA24_Susanna_Rokka/code-alongs/13_3_lancedb/vector_database')

In [2]:
db.uri

'/Users/susannarokka/Desktop/NBI_year_2/Repo/AI_Engineering_OPA24_Susanna_Rokka/code-alongs/13_3_lancedb/vector_database'

## create a table

In [5]:
import json

with open("data/animals_text_embeddings.json", "r") as file: #embeddings is a vector representation of text
    data = json.loads(file.read())

data


[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [7]:
table_animals = db.create_table("animals_text", exist_ok= True, data=data)
table_animals

LanceTable(name='animals_text', version=1, _conn=LanceDBConnection(uri='/Users/susannarokka/Desktop/NBI_year_2/Repo/AI_Engineering_OPA24_Susanna_Rokka/code-alongs/13_3_lancedb/vector_database'))

In [8]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


In [9]:
more_data = [    
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
    ]

table_animals.add(more_data)

AddResult(version=2)

In [10]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create an empty table, then delete it

In [12]:
# 1. add schema to table first - depends on pydantic model

from lancedb.pydantic import LanceModel

LanceModel

lancedb.pydantic.LanceModel

In [13]:
class JokeSchema(LanceModel):
    joke: str
    rating: int

db.create_table(name = "jokes", schema = JokeSchema, exist_ok = True)
db

LanceDBConnection(uri='/Users/susannarokka/Desktop/NBI_year_2/Repo/AI_Engineering_OPA24_Susanna_Rokka/code-alongs/13_3_lancedb/vector_database')

In [14]:
# to add data follow established schema

db.table_names()

['animals_text', 'jokes']

In [15]:
# to remove a table, simply drop_table("table_name")

db.drop_table("jokes")

In [16]:
db.table_names()

['animals_text']

## Open existing table

In [17]:
db.open_table("animals_text").head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

## Vector search in LanceDB

In [18]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [19]:
# query_vector is search query input

query_vector = [0.5, 0.2, 0.9]

table_animals.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.2673


## Embedding API

- idea: we want to put in text -> and it will automagically generate vector embeddings
- put in a query and it will automagically generate vector embeddings
- calculate closest distances

In [20]:
from lancedb.pydantic import Vector
from lancedb.embeddings import get_registry

model = get_registry().get("gemini-text").create(name = "gemini-embedding-001")
model

GeminiText(max_retries=7, name='gemini-embedding-001', query_task_type='retrieval_query', source_task_type='retrieval_document')

In [ ]:
# install google-genai 'uv pip install google-genai'

model.generate_embeddings("hello")


[[-0.010894743,
  -0.0028489418,
  0.00893425,
  -0.054541513,
  -0.007433747,
  0.017504206,
  0.014653749,
  0.02498823,
  0.01153613,
  -0.004031619,
  0.0019689437,
  -0.015896015,
  -0.0033653777,
  0.015975969,
  0.09177606,
  -0.02413529,
  0.026621163,
  -0.015998762,
  -0.009193494,
  -0.0032317825,
  0.017314918,
  -0.014104707,
  -0.000599844,
  -0.013850878,
  0.025816025,
  0.027195023,
  0.017755596,
  0.009800108,
  0.022374708,
  0.017374959,
  -0.0027653421,
  0.0067342324,
  0.00040601502,
  0.002778903,
  -0.004352005,
  0.011505135,
  0.022893922,
  -0.010107965,
  -0.0130854435,
  0.022383824,
  0.00030385726,
  -0.007134147,
  0.002239337,
  -0.0012981425,
  7.041567e-05,
  0.011082349,
  -0.005637078,
  -0.025727957,
  0.0059076254,
  0.0042295936,
  0.0017543633,
  0.011937581,
  -0.020759232,
  -0.14057352,
  -0.011109075,
  0.006916541,
  -0.0025381926,
  0.021703891,
  0.0076077846,
  -0.029725924,
  0.010393887,
  0.026465422,
  -0.014564507,
  -0.008960989,

In [25]:
import numpy as np
hello_embedding = np.array(model.generate_embeddings("hello"))
hello_embedding.shape

(5, 3072)

In [ ]:
model.ndims() # gives a different dimension than the actual? so hard code '3072' which is actual shape

768

In [27]:
class JokeModel(LanceModel):
    joke: str = model.SourceField() # marks a text as input to embedding function
    vector: Vector(3072) = model.VectorField() # a vector is the destination of a computed embedding

table_jokes = db.create_table("jokes", schema=JokeModel, exist_ok=True)
table_jokes

LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='/Users/susannarokka/Desktop/NBI_year_2/Repo/AI_Engineering_OPA24_Susanna_Rokka/code-alongs/13_3_lancedb/vector_database'))